<a href="https://colab.research.google.com/github/alfnihilus/alfnihilus/blob/main/this-is-code/alfnihilus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🔐 Konverter Sandi Morse
# @markdown ---
# @markdown ### Tentang Program Ini
# @markdown Program ini mengkonversi **teks biasa ↔ sandi Morse**.
# @markdown
# @markdown **Aturan Sandi Morse yang digunakan:**
# @markdown - Titik **`.`** = sinyal pendek
# @markdown - Garis **`-`** atau **`_`** = sinyal panjang (keduanya diterima)
# @markdown - Pemisah antar huruf: spasi tunggal
# @markdown - Pemisah antar kata: ` / `

In [ ]:
# @title ⚙️ Setup: Tabel & Fungsi Morse
# @markdown - .- -... . .-.. / -.- --- -.. . / -- --- .-. ... .
# Tabel encode: huruf → kode Morse
MORSE_TABLE = {
    'A': '.-',   'B': '-...', 'C': '-.-.',
    'D': '-..',  'E': '.',    'F': '..-.',
    'G': '--.',  'H': '....', 'I': '..',
    'J': '.---', 'K': '-.-',  'L': '.-..',
    'M': '--',   'N': '-.',   'O': '---',
    'P': '.--.', 'Q': '--.-', 'R': '.-.',
    'S': '...',  'T': '-',    'U': '..-',
    'V': '...-', 'W': '.--',  'X': '-..-',
    'Y': '-.--', 'Z': '--..',
    '0': '-----', '1': '.----', '2': '..---',
    '3': '...--', '4': '....-', '5': '.....',
    '6': '-....', '7': '--...', '8': '---..',
    '9': '----.',
    '.': '.-.-.-', ',': '--..--', '?': '..--..',
    '!': '-.-.--', '/': '-..-.', ' ': '/',
}

# Tabel decode: kode Morse → huruf
# Normalisasi: _ dianggap sama dengan -
DECODE_TABLE = {v: k for k, v in MORSE_TABLE.items() if k != ' '}

def normalisasi_morse(kode):
    """Mengganti _ dengan - agar konsisten."""
    return kode.replace('_', '-')

def teks_ke_morse(teks):
    """Mengkonversi teks biasa menjadi sandi Morse."""
    teks = teks.upper()
    hasil = []
    for kata in teks.split(' '):
        kode_kata = []
        for huruf in kata:
            if huruf in MORSE_TABLE:
                kode_kata.append(MORSE_TABLE[huruf])
            else:
                kode_kata.append('?')
        hasil.append(' '.join(kode_kata))
    return ' / '.join(hasil)

def morse_ke_teks(morse):
    """Mengkonversi sandi Morse menjadi teks biasa. Mendukung - dan _."""
    morse = normalisasi_morse(morse.strip())
    kata_list = morse.split(' / ')
    hasil = []
    for kata in kata_list:
        huruf_list = []
        for kode in kata.split(' '):
            if kode == '':
                continue
            huruf = DECODE_TABLE.get(kode, '?')
            huruf_list.append(huruf)
        hasil.append(''.join(huruf_list))
    return ' '.join(hasil)

print("✅ Fungsi morse siap digunakan!")

In [ ]:
# @title 🧪 Coba Konverter Morse
# @markdown Isi salah satu kolom di bawah, lalu jalankan sel ini.

input_teks   = ""  #@param {type:"string"}
input_morse  = ""            #@param {type:"string"}
tampilkan_tabel = False      #@param {type:"boolean"}

print("=" * 45)

if input_teks.strip():
    hasil = teks_ke_morse(input_teks)
    print(f"📝 Teks Asli  : {input_teks.upper()}")
    print(f"📡 Sandi Morse: {hasil}")

if input_morse.strip():
    hasil = morse_ke_teks(input_morse)
    print(f"📡 Sandi Morse: {input_morse}")
    print(f"📝 Teks Asli  : {hasil}")

if tampilkan_tabel:
    print("\n📖 Tabel Referensi Morse:")
    print("-" * 30)
    items = [(k, v) for k, v in MORSE_TABLE.items() if k != ' ']
    for i in range(0, len(items), 3):
        baris = items[i:i+3]
        print("  ".join(f"{k} → {v:<7}" for k, v in baris))

print("=" * 45)

In [ ]:
# @title 🔊 Generator Suara Morse
# @markdown Konversi teks ke sandi Morse **sekaligus membunyikannya!**
# @markdown
# @markdown - 🔵 Titik `.` = beep **pendek** (100ms)
# @markdown - 🟠 Garis `-` atau `_` = beep **panjang** (300ms)
# @markdown - Frekuensi nada bisa diatur di bawah.

import numpy as np
from IPython.display import Audio, display

teks_suara  = ""   #@param {type:"string"}
frekuensi   = 700     #@param {type:"slider", min:300, max:1200, step:50}
kecepatan   = 1     #@param {type:"slider", min:0.5, max:2.0, step:0.1}

# ── Parameter waktu (dalam detik, disesuaikan kecepatan) ──────────────
SAMPLE_RATE  = 44100
DIT          = 0.10 / kecepatan   # titik
DAH          = 0.30 / kecepatan   # garis
JEDA_SIMBOL  = 0.08 / kecepatan   # jeda antar simbol dalam 1 huruf
JEDA_HURUF   = 0.20 / kecepatan   # jeda antar huruf
JEDA_KATA    = 0.50 / kecepatan   # jeda antar kata

def buat_nada(durasi, freq=frekuensi, sr=SAMPLE_RATE):
    """Membuat gelombang sinus dengan fade in/out agar tidak 'klik'."""
    t = np.linspace(0, durasi, int(sr * durasi), endpoint=False)
    gelombang = 0.5 * np.sin(2 * np.pi * freq * t)
    # Fade in & out 10ms supaya tidak terdengar 'klik'
    fade = int(sr * 0.01)
    gelombang[:fade]  *= np.linspace(0, 1, fade)
    gelombang[-fade:] *= np.linspace(1, 0, fade)
    return gelombang

def buat_senyap(durasi, sr=SAMPLE_RATE):
    return np.zeros(int(sr * durasi))

def teks_ke_audio(teks):
    teks = teks.upper()
    audio = np.array([])
    morse_visual = []

    for i, kata in enumerate(teks.split(' ')):
        kode_kata = []
        for j, huruf in enumerate(kata):
            if huruf not in MORSE_TABLE:
                continue
            kode = MORSE_TABLE[huruf]
            kode_kata.append(kode)
            simbol_list = list(kode)
            for k, simbol in enumerate(simbol_list):
                if simbol == '.':
                    audio = np.concatenate([audio, buat_nada(DIT)])
                elif simbol in ('-', '_'):
                    audio = np.concatenate([audio, buat_nada(DAH)])
                # Jeda antar simbol (kecuali simbol terakhir di huruf)
                if k < len(simbol_list) - 1:
                    audio = np.concatenate([audio, buat_senyap(JEDA_SIMBOL)])
            # Jeda antar huruf (kecuali huruf terakhir di kata)
            if j < len(kata) - 1:
                audio = np.concatenate([audio, buat_senyap(JEDA_HURUF)])
        morse_visual.append(' '.join(kode_kata))
        # Jeda antar kata
        if i < len(teks.split(' ')) - 1:
            audio = np.concatenate([audio, buat_senyap(JEDA_KATA)])

    return audio, ' / '.join(morse_visual)

# ── Jalankan ──────────────────────────────────────────────────────────
audio_data, morse_output = teks_ke_audio(teks_suara)

print("=" * 45)
print(f"📝 Teks Asli   : {teks_suara.upper()}")
print(f"📡 Sandi Morse : {morse_output}")
print(f"🔊 Frekuensi   : {frekuensi} Hz")
print(f"⚡ Kecepatan   : {kecepatan}x")
print("=" * 45)
print("▶️  Memutar audio...")

display(Audio(audio_data, rate=SAMPLE_RATE, autoplay=True))